# Étape 6 — Préparation de l'expérience de prédiction de liens

Ce notebook prépare un benchmark reproductible et sans fuite d'information à partir du graphe
biparti **original**. Il crée un split positif train/validation/test, un graphe d'entraînement et
des négatifs équilibrés. Il ne calcule aucun score et n'exécute aucun algorithme de link prediction.

## 1. Chargement du graphe complet

Nous chargeons `cv_job_graph.pkl`, produit à l'Étape 2. Nous n'utilisons pas la version enrichie par
les communautés, car ces communautés ont été calculées avec toutes les arêtes et révéleraient donc
indirectement les liens que nous allons cacher. Les contrôles ci-dessous identifient simplement le
bon graphe ; ils ne répètent pas l'analyse structurelle.

In [1]:
from pathlib import Path
import json
import pickle
import random

import networkx as nx
import numpy as np
import pandas as pd

SEED = 42
ROOT = Path("..") if Path.cwd().name == "notebooks" else Path(".")
PROCESSED_DIR = ROOT / "data" / "processed"
RESULTS_DIR = ROOT / "results"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

with (PROCESSED_DIR / "cv_job_graph.pkl").open("rb") as file:
    G_full = pickle.load(file)

cv_nodes = sorted(
    node for node, data in G_full.nodes(data=True) if data.get("node_type") == "CV"
)
job_nodes = sorted(
    node for node, data in G_full.nodes(data=True) if data.get("node_type") == "Job"
)
cv_set = set(cv_nodes)
job_set = set(job_nodes)

assert G_full.number_of_nodes() == 12_500
assert G_full.number_of_edges() == 75_000
assert len(cv_nodes) == 10_000
assert len(job_nodes) == 2_500
assert nx.is_bipartite(G_full)
assert cv_set.isdisjoint(job_set)

display(pd.Series({
    "n_nodes": G_full.number_of_nodes(),
    "n_edges": G_full.number_of_edges(),
    "n_cv": len(cv_nodes),
    "n_jobs": len(job_nodes),
    "is_bipartite": nx.is_bipartite(G_full),
}, name="valeur").to_frame())

,valeur
n_nodes,12500
n_edges,75000
n_cv,10000
n_jobs,2500
is_bipartite,True


## 2. Pourquoi cacher volontairement de vrais liens ?

La prédiction de liens doit être évaluée sur des liens que la méthode n'a pas vus. Supposons que le
graphe complet contienne `CV1—JobA` et `CV1—JobB`. Nous pouvons cacher `CV1—JobB`, donner au futur
algorithme le graphe ne contenant que `CV1—JobA`, puis vérifier s'il attribue un score élevé à la
paire cachée. Le lien caché reste un positif connu dans notre **vérité terrain**, mais il est absent
de la structure fournie à la méthode.

Nous séparons les positifs en :

- **train** : liens conservés dans le graphe d'entraînement ;
- **validation** : liens cachés qui serviront à choisir les paramètres ;
- **test** : liens cachés réservés à la comparaison finale.

Toutes les futures méthodes devront partager exactement ce split. Sinon, leurs scores seraient
évalués sur des difficultés différentes et ne seraient pas directement comparables.

## 3. Orientation canonique des paires CV–Job

NetworkX stocke une arête non orientée sans garantir que le CV apparaît en première position. Pour
les fichiers expérimentaux, nous convertissons donc toujours une arête en
`(resume_id, job_id)`. Cette représentation canonique évite qu'une même paire soit considérée deux
fois sous les formes `(CV, Job)` et `(Job, CV)`.

In [2]:
def canonical_pair(u, v):
    if u in cv_set and v in job_set:
        return (u, v)
    if v in cv_set and u in job_set:
        return (v, u)
    raise ValueError(f"L'arête ({u}, {v}) n'est pas une paire CV–Job.")


full_positive = {canonical_pair(u, v) for u, v in G_full.edges()}
assert len(full_positive) == 75_000

## 4. Split positif warm-start 80 % / 10 % / 10 %

Un split purement aléatoire pourrait retirer toutes les arêtes d'un CV peu connecté. Ce nœud
deviendrait artificiellement isolé dans le train et aucune méthode topologique ne disposerait
d'information sur lui. Nous réalisons donc une évaluation **warm-start** : validation et test
contiennent seulement des liens entre nœuds déjà observés et encore connectés dans le train.

Algorithme reproductible :

1. mélanger les 75 000 arêtes avec la seed 42 ;
2. partir des degrés du graphe complet ;
3. parcourir les arêtes mélangées ;
4. cacher une arête seulement si ses deux extrémités garderont un degré résiduel d'au moins 1 ;
5. arrêter après 15 000 retraits, puis mélanger ces retraits et les partager en 7 500 validation et
   7 500 test.

Les 60 000 autres arêtes deviennent les positives train. Il s'agit uniquement d'un warm-start ;
aucune expérience cold-start n'est ajoutée.

In [3]:
TARGET_HIDDEN = 15_000
TARGET_VAL = 7_500
TARGET_TEST = 7_500

rng_positive = random.Random(SEED)
candidate_edges = sorted(full_positive)
rng_positive.shuffle(candidate_edges)
residual_degree = dict(G_full.degree())
hidden_edges = []

for resume_id, job_id in candidate_edges:
    if residual_degree[resume_id] > 1 and residual_degree[job_id] > 1:
        hidden_edges.append((resume_id, job_id))
        residual_degree[resume_id] -= 1
        residual_degree[job_id] -= 1
        if len(hidden_edges) == TARGET_HIDDEN:
            break

if len(hidden_edges) != TARGET_HIDDEN:
    raise RuntimeError(
        f"Seulement {len(hidden_edges)} arêtes peuvent être cachées sans créer d'isolés."
    )

rng_positive.shuffle(hidden_edges)
positive_val = set(hidden_edges[:TARGET_VAL])
positive_test = set(hidden_edges[TARGET_VAL:TARGET_VAL + TARGET_TEST])
positive_train = full_positive - positive_val - positive_test

print("Positifs train :", len(positive_train))
print("Positifs validation :", len(positive_val))
print("Positifs test :", len(positive_test))

Positifs train : 60000
Positifs validation : 7500
Positifs test : 7500


## 5. Construction du graphe d'entraînement

`G_train` est une copie indépendante du graphe original : elle conserve les 12 500 nœuds et tous
leurs attributs, mais ne garde que les 60 000 arêtes positives train. Les 15 000 liens de validation
et test doivent en être absents. Ce graphe deviendra la seule référence topologique autorisée pour
les futures méthodes de prédiction de liens.

In [4]:
G_train = nx.Graph()
G_train.graph.update(G_full.graph)
G_train.add_nodes_from((node, data.copy()) for node, data in G_full.nodes(data=True))
G_train.add_edges_from(sorted(positive_train))

assert G_train.number_of_nodes() == G_full.number_of_nodes()
assert G_train.number_of_edges() == 60_000
assert all(not G_train.has_edge(*pair) for pair in positive_val | positive_test)

with (PROCESSED_DIR / "cv_job_graph_link_train.pkl").open("wb") as file:
    pickle.dump(G_train, file, protocol=pickle.HIGHEST_PROTOCOL)

print("G_train sauvegardé avec", G_train.number_of_edges(), "arêtes.")

G_train sauvegardé avec 60000 arêtes.


## 6. Effet du split sur les isolés et les degrés

Un nœud isolé a un degré nul. Certains CV étaient déjà isolés dans le graphe complet ; le split ne
peut pas leur inventer des liens. En revanche, aucun nœud initialement connecté ne doit devenir
isolé à cause du split. Nous vérifions cette propriété et décrivons brièvement les degrés train par
partition. Ces statistiques contrôlent le split sans refaire l'analyse structurelle.

In [5]:
isolates_full = set(nx.isolates(G_full))
isolates_train = set(nx.isolates(G_train))
initially_connected = {node for node, degree in G_full.degree() if degree > 0}
new_isolates = isolates_train & initially_connected

degree_cv_train = pd.Series(
    [G_train.degree(node) for node in cv_nodes], name="CV"
)
degree_job_train = pd.Series(
    [G_train.degree(node) for node in job_nodes], name="Job"
)
degree_control = pd.DataFrame({
    "CV": {
        "minimum": int(degree_cv_train.min()),
        "moyenne": float(degree_cv_train.mean()),
        "médiane": float(degree_cv_train.median()),
    },
    "Job": {
        "minimum": int(degree_job_train.min()),
        "moyenne": float(degree_job_train.mean()),
        "médiane": float(degree_job_train.median()),
    },
})

print("Isolés dans G_full :", len(isolates_full))
print("Isolés dans G_train :", len(isolates_train))
print("Nouveaux isolés créés :", len(new_isolates))
display(degree_control)
assert not new_isolates

Isolés dans G_full : 29
Isolés dans G_train : 29
Nouveaux isolés créés : 0


,CV,Job
minimum,0.0,14.0
moyenne,6.0,24.0
médiane,6.0,24.0


## 7. Positifs, négatifs et déséquilibre naturel

Un **positif** est une paire CV–Job parmi les 75 000 matches connus. Un **négatif d'évaluation** est
une paire CV–Job qui n'appartient à aucun de ces matches connus. L'espace biparti contient

$$10\,000 \times 2\,500 = 25\,000\,000$$

paires possibles, mais seulement 75 000 positives, soit environ 0,3 %. Le problème naturel est donc
très déséquilibré. Comme présenté dans le cours, le **downsampling** de la classe majoritaire permet
de construire un benchmark équilibré et maniable.

Précaution d'interprétation : dans un dataset réel incomplet, une absence de lien peut signifier
« non observé » plutôt que « réellement incompatible ». Ici, nous appelons négatifs les non-matches
selon la vérité terrain disponible, sans prétendre prouver une incompatibilité absolue.

In [6]:
n_possible_cv_job_pairs = len(cv_nodes) * len(job_nodes)
positive_rate_full_space = len(full_positive) / n_possible_cv_job_pairs

display(pd.Series({
    "paires CV–Job possibles": n_possible_cv_job_pairs,
    "positifs connus": len(full_positive),
    "non-positifs connus": n_possible_cv_job_pairs - len(full_positive),
    "taux positif": positive_rate_full_space,
    "taux positif (%)": 100 * positive_rate_full_space,
}, name="valeur").to_frame())

,valeur
paires CV–Job possibles,2.500000e+07
positifs connus,7.500000e+04
non-positifs connus,2.492500e+07
taux positif,3.000000e-03
taux positif (%),3.000000e-01


## 8. Règle anti-leakage pour les négatifs

Une paire absente de `G_train` n'est pas nécessairement négative : les 15 000 vrais liens de
validation et test en sont volontairement absents. Les considérer comme négatifs apprendrait au
futur modèle une contradiction et contaminerait l'évaluation. La blacklist utilisée pour le
negative sampling est donc l'ensemble `full_positive` des **75 000 positives complètes**, jamais la
seule liste des arêtes train.

## 9. Negative sampling efficace et reproductible

Nous avons besoin de 75 000 négatifs parmi près de 25 millions de possibilités. Matérialiser toutes
les paires gaspillerait de la mémoire. À chaque tirage, nous choisissons aléatoirement un CV et un
Job, rejetons la paire si elle appartient aux positives complètes ou si elle a déjà été tirée, puis
continuons jusqu'à obtenir l'effectif voulu. Un `set` permet des tests d'appartenance rapides en
temps moyen constant. La seed 42 rend les tirages reproductibles.

In [7]:
TARGET_NEGATIVES = 75_000
rng_negative = random.Random(SEED)
all_negatives = set()

while len(all_negatives) < TARGET_NEGATIVES:
    pair = (rng_negative.choice(cv_nodes), rng_negative.choice(job_nodes))
    if pair not in full_positive:
        all_negatives.add(pair)

negative_list = sorted(all_negatives)
rng_negative.shuffle(negative_list)
negative_train = set(negative_list[:60_000])
negative_val = set(negative_list[60_000:67_500])
negative_test = set(negative_list[67_500:75_000])

print("Négatifs train :", len(negative_train))
print("Négatifs validation :", len(negative_val))
print("Négatifs test :", len(negative_test))

Négatifs train : 60000
Négatifs validation : 7500
Négatifs test : 7500


## 10. Création des trois datasets de paires

Chaque fichier associe autant de négatifs que de positifs : c'est un benchmark équilibré obtenu par
downsampling des non-matches. La fonction ci-dessous crée les colonnes `resume_id`, `job_id` et
`label`, concatène les deux classes, puis mélange les lignes avec la seed 42. Le tri préalable des
sets garantit que le résultat ne dépend pas de leur ordre interne en mémoire.

In [8]:
def make_pair_dataset(positives, negatives, seed=SEED):
    positive_df = pd.DataFrame(
        sorted(positives), columns=["resume_id", "job_id"]
    ).assign(label=1)
    negative_df = pd.DataFrame(
        sorted(negatives), columns=["resume_id", "job_id"]
    ).assign(label=0)
    return (
        pd.concat([positive_df, negative_df], ignore_index=True)
        .sample(frac=1, random_state=seed)
        .reset_index(drop=True)
    )


train_pairs_df = make_pair_dataset(positive_train, negative_train)
val_pairs_df = make_pair_dataset(positive_val, negative_val)
test_pairs_df = make_pair_dataset(positive_test, negative_test)

train_pairs_df.to_csv(RESULTS_DIR / "step6_link_train_pairs.csv", index=False)
val_pairs_df.to_csv(RESULTS_DIR / "step6_link_val_pairs.csv", index=False)
test_pairs_df.to_csv(RESULTS_DIR / "step6_link_test_pairs.csv", index=False)

display(pd.DataFrame({
    "train": train_pairs_df["label"].value_counts().sort_index(),
    "validation": val_pairs_df["label"].value_counts().sort_index(),
    "test": test_pairs_df["label"].value_counts().sort_index(),
}).rename_axis("label"))

,train,validation,test
label,,,
0,60000,7500,7500
1,60000,7500,7500


## 11. Assertions anti-leakage

Ces assertions constituent le contrat expérimental des étapes futures. Elles empêchent qu'un lien
caché soit visible dans le graphe train, qu'un positif soit étiqueté négatif, qu'une paire soit
réutilisée entre splits ou qu'un nœud connecté soit artificiellement isolé. En cas d'échec, le
notebook s'arrête immédiatement : continuer produirait des performances trompeuses.

In [9]:
def pair_is_cv_job(pair):
    resume_id, job_id = pair
    return resume_id in cv_set and job_id in job_set


# Positifs : tailles, disjonction et reconstruction exacte.
assert len(positive_train) == 60_000
assert len(positive_val) == 7_500
assert len(positive_test) == 7_500
assert positive_train.isdisjoint(positive_val)
assert positive_train.isdisjoint(positive_test)
assert positive_val.isdisjoint(positive_test)
assert positive_train | positive_val | positive_test == full_positive

# Graphe train : uniquement les positifs train.
train_graph_pairs = {canonical_pair(u, v) for u, v in G_train.edges()}
assert G_train.number_of_edges() == 60_000
assert train_graph_pairs == positive_train
assert all(not G_train.has_edge(*pair) for pair in positive_val | positive_test)

# Négatifs : aucun positif complet, aucun doublon inter-split.
assert len(negative_train) == 60_000
assert len(negative_val) == 7_500
assert len(negative_test) == 7_500
assert (negative_train | negative_val | negative_test).isdisjoint(full_positive)
assert negative_train.isdisjoint(negative_val)
assert negative_train.isdisjoint(negative_test)
assert negative_val.isdisjoint(negative_test)

# Toutes les paires respectent les deux partitions du graphe.
all_experimental_pairs = (
    full_positive | negative_train | negative_val | negative_test
)
assert all(pair_is_cv_job(pair) for pair in all_experimental_pairs)

# Warm-start et fichiers finaux.
assert not new_isolates
for frame, expected_size in [
    (train_pairs_df, 120_000),
    (val_pairs_df, 15_000),
    (test_pairs_df, 15_000),
]:
    assert len(frame) == expected_size
    assert frame.columns.tolist() == ["resume_id", "job_id", "label"]
    assert not frame.duplicated(["resume_id", "job_id"]).any()
    assert set(frame["label"]) == {0, 1}

print("Toutes les assertions anti-leakage sont passées.")

Toutes les assertions anti-leakage sont passées.


## 12. Rôle de chaque split pour la suite

### Train

`G_train` est la seule structure topologique autorisée pour calculer les futurs scores de link
prediction. Les paires train pourront aussi servir si un modèle supervisé ou hybride est entraîné.

### Validation

La validation servira à choisir, par exemple, un seuil de similarité, un hyperparamètre de Katz, un
seuil de décision ou les paramètres d'un modèle hybride.

### Test

Le test doit rester intact jusqu'à la comparaison finale. Aucune décision méthodologique ne devra
être prise en regardant ses performances, sous peine d'adapter indirectement la méthode au test.

## 13. Fuite provenant des anciennes features structurelles

Les degrés, PageRank, betweenness et `community_global` calculés auparavant sur `G_full` restent
valides pour l'analyse descriptive du réseau. Ils ne doivent toutefois pas être utilisés tels quels
pour prédire les liens validation/test : leur calcul a vu les arêtes maintenant censées être
cachées. Si des features structurelles sont nécessaires plus tard, elles devront être recalculées
exclusivement depuis `G_train`.

Les features fondées seulement sur le texte ou les attributs propres des nœuds ne voient pas les
arêtes et ne subissent pas cette même fuite topologique. Nous documentons cette règle sans recalculer
de communautés à cette étape.

## 14. Plan d'évaluation futur — sans calcul de score maintenant

Les futures méthodes produiront un score $s(CV,Job)$. Comme dans le cours, les paires pourront être
classées par score décroissant pour recommander les meilleurs liens. Toutes les méthodes utiliseront
le même split et seront comparées avec ROC-AUC, Precision, Recall et F1 ; l'aspect ranking utilisera
aussi Precision@K et Recall@K. Aucune de ces métriques n'est calculée ici puisqu'aucun score de
prédiction n'existe encore.

## 15. Résumé reproductible

Le JSON suivant mémorise la seed, la stratégie warm-start, les effectifs, les isolés et les contrôles
anti-leakage. Il permettra aux étapes futures de vérifier qu'elles utilisent bien le benchmark
officiel préparé ici.

In [10]:
positive_sets_disjoint = (
    positive_train.isdisjoint(positive_val)
    and positive_train.isdisjoint(positive_test)
    and positive_val.isdisjoint(positive_test)
)
negative_sets_disjoint = (
    negative_train.isdisjoint(negative_val)
    and negative_train.isdisjoint(negative_test)
    and negative_val.isdisjoint(negative_test)
)
anti_leakage_confirmed = (
    all(not G_train.has_edge(*pair) for pair in positive_val | positive_test)
    and (negative_train | negative_val | negative_test).isdisjoint(full_positive)
)

split_summary = {
    "seed": SEED,
    "split_strategy": (
        "80/10/10 positive edge split; shuffled greedy holdout accepted only when both "
        "endpoints retain residual train degree >= 1"
    ),
    "warm_start": True,
    "n_possible_cv_job_pairs": int(n_possible_cv_job_pairs),
    "n_full_positive_edges": int(len(full_positive)),
    "positive_rate_full_space": float(positive_rate_full_space),
    "n_train_positive": int(len(positive_train)),
    "n_val_positive": int(len(positive_val)),
    "n_test_positive": int(len(positive_test)),
    "n_train_negative": int(len(negative_train)),
    "n_val_negative": int(len(negative_val)),
    "n_test_negative": int(len(negative_test)),
    "n_train_graph_edges": int(G_train.number_of_edges()),
    "n_isolates_full": int(len(isolates_full)),
    "n_isolates_train": int(len(isolates_train)),
    "n_new_isolates_created": int(len(new_isolates)),
    "positive_sets_disjoint": bool(positive_sets_disjoint),
    "negative_sets_disjoint": bool(negative_sets_disjoint),
    "positive_union_reconstructs_full_graph": bool(
        positive_train | positive_val | positive_test == full_positive
    ),
    "validation_test_absent_from_train_graph": True,
    "negative_blacklist_is_full_positive_graph": True,
    "no_negative_is_known_positive": True,
    "anti_leakage_confirmed": bool(anti_leakage_confirmed),
    "all_pairs_are_cv_job": True,
    "all_assertions_passed": True,
    "scope_confirmation": "No link-prediction algorithm or score was executed.",
}
(RESULTS_DIR / "step6_link_split_summary.json").write_text(
    json.dumps(split_summary, indent=2, ensure_ascii=False), encoding="utf-8"
)
display(pd.Series(split_summary, name="valeur").to_frame())

,valeur
seed,42
split_strategy,80/10/10 positive edge split; shuffled greedy ...
warm_start,True
n_possible_cv_job_pairs,25000000
n_full_positive_edges,75000
positive_rate_full_space,0.003
n_train_positive,60000
n_val_positive,7500
n_test_positive,7500
n_train_negative,60000


## Interprétation et conclusion

- Nous cachons 15 000 vrais liens afin que les futures méthodes soient évaluées sur des positives
  qu'elles n'ont pas vues : 7 500 servent à la validation et 7 500 au test final.
- Les 60 000 autres liens restent dans `G_train`. Le retrait warm-start garantit que chaque nœud
  initialement connecté conserve au moins une arête d'entraînement.
- Les négatifs sont définis par rapport aux 75 000 positives du graphe complet, jamais par simple
  absence dans `G_train`, car validation et test sont précisément des positives absentes du train.
- L'espace naturel contient 25 millions de paires, dont seulement 0,3 % de positives. Ce fort
  déséquilibre est important pour l'interprétation réelle. Le benchmark utilise néanmoins autant de
  négatifs que de positifs dans chaque split afin de comparer proprement les méthodes sur un volume
  maîtrisé ; ce downsampling ne change pas le taux naturel documenté.
- Toutes les futures approches — sémantiques, classiques ou hybrides — devront utiliser exactement
  ces CSV et `G_train`. Ce dernier devient l'unique référence topologique : toute feature
  structurelle destinée à la link prediction devra être recalculée depuis ce graphe.

Aucun embedding, score de similarité, Common Neighbors, Adamic-Adar, Katz, classifieur ou autre
algorithme de prédiction de liens n'est exécuté dans ce notebook.